In [ ]:
"""
Project 02: Predictive Maintenance
Step 03: Evaluation & Metrics
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import time
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import train_test_split

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 03: Evaluation & Metrics")
print("=" * 60)
start_time = time.time()

In [ ]:
# 1. Load Data & Models
print("\n[1/5] Loading data and models...")
df = pd.read_csv(os.path.join(PROCESSED, "processed_data.csv"))
X = df.drop(columns=["Target"])

In [ ]:
with open(os.path.join(MODELS, "label_encoder.pkl"), "rb") as f:
    le = pickle.load(f)
y_encoded = le.transform(df["Target"])

In [ ]:
_, X_test, _, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [ ]:
model_names = ["decision_tree", "random_forest", "xgboost"]
models = {}
for name in model_names:
    path = os.path.join(MODELS, f"{name}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            models[name] = pickle.load(f)
        print(f"  Loaded: {name}.pkl")
    else:
        print(f"  WARNING: {name}.pkl not found")

In [ ]:
print(f"  Test samples: {len(y_test)}")
print(f"  Classes: {list(le.classes_)}")

In [ ]:
# 2. Compute Metrics
print("\n[2/5] Computing evaluation metrics for all models...")
all_metrics = {}
all_preds = {}

In [ ]:
for name, model in tqdm(models.items(), desc="  Evaluating models"):
    preds = model.predict(X_test)
    all_preds[name] = preds

    acc = accuracy_score(y_test, preds)
    prec_macro = precision_score(y_test, preds, average="macro", zero_division=0)
    rec_macro = recall_score(y_test, preds, average="macro", zero_division=0)
    f1_macro = f1_score(y_test, preds, average="macro", zero_division=0)
    prec_weighted = precision_score(y_test, preds, average="weighted", zero_division=0)
    rec_weighted = recall_score(y_test, preds, average="weighted", zero_division=0)
    f1_weighted = f1_score(y_test, preds, average="weighted", zero_division=0)

    all_metrics[name] = {
        "Accuracy": acc,
        "Precision (Macro)": prec_macro,
        "Recall (Macro)": rec_macro,
        "F1 (Macro)": f1_macro,
        "Precision (Weighted)": prec_weighted,
        "Recall (Weighted)": rec_weighted,
        "F1 (Weighted)": f1_weighted,
    }

    print(f"\n  {name.replace('_', ' ').title()}:")
    print(f"    Accuracy:  {acc:.4f}")
    print(f"    Precision: {prec_macro:.4f} (macro) | {prec_weighted:.4f} (weighted)")
    print(f"    Recall:    {rec_macro:.4f} (macro) | {rec_weighted:.4f} (weighted)")
    print(f"    F1 Score:  {f1_macro:.4f} (macro) | {f1_weighted:.4f} (weighted)")

In [ ]:
# 3. Save Metrics
print("\n[3/5] Saving metrics...")
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "Model"
metrics_df.to_csv(os.path.join(MODELS, "performance_metrics.csv"))

In [ ]:
with open(os.path.join(MODELS, "performance_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=4)

In [ ]:
print(f"  Saved: performance_metrics.csv, performance_metrics.json")

In [ ]:
# 4. Classification Reports
print("\n[4/5] Generating classification reports...")
for name, preds in all_preds.items():
    print(f"\n  --- {name.replace('_', ' ').title()} Classification Report ---")
    report = classification_report(
        y_test, preds, target_names=le.classes_, zero_division=0
    )
    print(report)

In [ ]:
# 5. Generate Charts
print("\n[5/5] Generating evaluation charts...")

In [ ]:
# Chart 1: Confusion Matrices
for name, preds in tqdm(all_preds.items(), desc="  Confusion matrices"):
    cm = confusion_matrix(y_test, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        ax=ax,
    )
    ax.set_title(f"{name.replace('_', ' ').title()} - Confusion Matrix", fontsize=14)
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, f"{name}_confusion_matrix.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 2: Normalized Confusion Matrices
for name, preds in tqdm(all_preds.items(), desc="  Normalized confusion matrices"):
    cm = confusion_matrix(y_test, preds, normalize="true")
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f",
        cmap="Oranges",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        ax=ax,
    )
    ax.set_title(
        f"{name.replace('_', ' ').title()} - Normalized Confusion Matrix", fontsize=14
    )
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(
        os.path.join(CHARTS, f"{name}_confusion_matrix_normalized.png"), dpi=150
    )
    plt.close()

In [ ]:
# Chart 3: Model Performance Comparison Bar Chart
for i in tqdm(range(1), desc="  Performance comparison chart"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    metric_groups = [
        ("Accuracy", "Accuracy"),
        ("Precision (Macro)", "Precision (Macro)"),
        ("Recall (Macro)", "Recall (Macro)"),
    ]
    colors = ["#3498db", "#e74c3c", "#2ecc71"]
    for idx, (metric, title) in enumerate(metric_groups):
        vals = [all_metrics[m][metric] for m in model_names]
        labels = [m.replace("_", " ").title() for m in model_names]
        bars = axes[idx].bar(labels, vals, color=colors, edgecolor="black")
        axes[idx].set_title(title, fontsize=12)
        axes[idx].set_ylim(0, 1.05)
        for bar, val in zip(bars, vals):
            axes[idx].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{val:.4f}",
                ha="center",
                fontsize=9,
            )
        axes[idx].tick_params(axis="x", rotation=15)
    plt.suptitle("Model Performance Comparison", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(
        os.path.join(CHARTS, "model_performance_comparison.png"),
        dpi=150,
        bbox_inches="tight",
    )
    plt.close()

In [ ]:
# Chart 4: All Metrics Heatmap
for i in tqdm(range(1), desc="  Metrics heatmap"):
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(
        metrics_df,
        annot=True,
        fmt=".4f",
        cmap="YlOrRd",
        ax=ax,
        linewidths=0.5,
        cbar_kws={"label": "Score"},
    )
    ax.set_title("All Models - Metrics Heatmap", fontsize=14)
    ax.set_yticklabels([n.replace("_", " ").title() for n in model_names], rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "metrics_heatmap.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 5: Per-Class Recall (Critical for imbalanced failure detection)
for i in tqdm(range(1), desc="  Per-class recall chart"):
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(le.classes_))
    width = 0.25
    colors_models = ["#3498db", "#e74c3c", "#2ecc71"]
    for idx, (name, preds) in enumerate(all_preds.items()):
        per_class_recall = []
        for cls_idx in range(len(le.classes_)):
            mask = y_test == cls_idx
            if mask.sum() > 0:
                rec = (preds[mask] == cls_idx).sum() / mask.sum()
            else:
                rec = 0
            per_class_recall.append(rec)
        ax.bar(
            x + idx * width,
            per_class_recall,
            width,
            label=name.replace("_", " ").title(),
            color=colors_models[idx],
            edgecolor="black",
        )
    ax.set_xlabel("Failure Type")
    ax.set_ylabel("Recall")
    ax.set_title("Per-Class Recall (Critical: Detecting Rare Failures)", fontsize=14)
    ax.set_xticks(x + width)
    ax.set_xticklabels(le.classes_, rotation=45, ha="right")
    ax.legend()
    ax.set_ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "per_class_recall.png"), dpi=150)
    plt.close()

In [ ]:
# Chart 6: Best Model Summary Dashboard
for i in tqdm(range(1), desc="  Summary dashboard"):
    best_model = max(all_metrics, key=lambda m: all_metrics[m]["F1 (Macro)"])
    best_f1 = all_metrics[best_model]["F1 (Macro)"]
    best_acc = all_metrics[best_model]["Accuracy"]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis("off")
    summary_text = (
        f"PROJECT 02: PREDICTIVE MAINTENANCE\n"
        f"{'=' * 50}\n\n"
        f"Best Model: {best_model.replace('_', ' ').title()}\n"
        f"Accuracy: {best_acc:.4f}\n"
        f"F1 Score (Macro): {best_f1:.4f}\n\n"
        f"Models Trained: {len(models)}\n"
        f"Features: {len(X.columns)}\n"
        f"Test Samples: {len(y_test)}\n"
        f"Classes: {len(le.classes_)}\n\n"
        f"Key Insight: Cost-Sensitive Learning\n"
        f"handles 96.5% class imbalance by\n"
        f"penalizing missed failures heavily."
    )
    ax.text(
        0.1,
        0.5,
        summary_text,
        fontsize=14,
        fontfamily="monospace",
        verticalalignment="center",
        bbox=dict(boxstyle="round", facecolor="#1a1a2e", alpha=0.8),
        color="#f39c12",
    )
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "summary_dashboard.png"), dpi=150)
    plt.close()

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 03 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")